In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import torch
import numpy as np

from src.utils import (
    get_args,
    set_seed,
    get_datesets_and_loaders,
    get_trained_VAE,
    get_trained_VAE_with_domain_classifier,
    get_trained_classifier,
    get_trained_classifier_Base,
    test_model,
    prepare_report,
    run_all_senario
)
from src.tupl import run_tupl

/home/asad/workspace/DomainProject/changeDomain/notebooks/effective-gzsda/gzsda/src/utils.py:3: UserWarning: A NumPy version >=1.22.4 and <2.3.0 is required for this version of SciPy (detected version 2.3.1)
  import scipy


In [3]:
DOMAIN_SET =['regu','xray']
DATA_DIR = './data/XrayBaggage20/'
DATASET_DETAILS = {
    "prefix": 'XrayDataset-',
    "suffix": '-resnet101-noft.mat',
    "resnet_feature": 'resnet101_features',
    "split_file_name": 'instanceSplit_xrayDataset_unseen10.mat',
}
NUM_LABELS=20

In [4]:
import json
from pathlib import Path

RESULT_OBJ_PATH = "./result/json/xray.json"
RESULT_CSV_PATH = "./result/csv/xray.csv"
path = Path(RESULT_OBJ_PATH)

if path.exists():
    with path.open("r", encoding="utf-8") as f:
        result = json.load(f)
else:
    result = {}

result.keys()

dict_keys(['base', 'CCVAE', 'our_GRE', 'our0', 'TUPL'])

In [5]:
base = "base"
CCVAE = "CCVAE"
our0 = "our0"
our_GRE = "our_GRE"
tupl = "TUPL"

# clear last result
# result.pop(base, None)
# result.pop(CCVAE, None)
# result.pop(our0, None)
# result.pop(our_GRE, None)
# result.pop(tupl, None)

## Base

In [6]:
def main_base(args):
    set_seed(args)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    datasets, data_loaders = get_datesets_and_loaders(
        args=args,
        DOMAIN_SET=DOMAIN_SET,
        DATA_DIR=DATA_DIR,
        DATASET_DETAILS=DATASET_DETAILS)

    classifier = get_trained_classifier_Base(
        data_loaders=data_loaders,
        NUM_LABELS=NUM_LABELS,
        device=device)

    return test_model(classifier, datasets['test'], data_loaders['test'], device)

In [7]:
if base not in result:
    result[base] = run_all_senario(main_base, DOMAIN_SET)

# GZSDA

In [8]:
def main_gzsda(args):
    set_seed(args)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    datasets, data_loaders = get_datesets_and_loaders(
        args=args,
        DOMAIN_SET=DOMAIN_SET,
        DATA_DIR=DATA_DIR,
        DATASET_DETAILS=DATASET_DETAILS)

    vae = get_trained_VAE(
        data_loaders=data_loaders,
        args=args,
        device=device)

    classifier = get_trained_classifier(
        data_loaders=data_loaders,
        vae=vae,
        NUM_LABELS=NUM_LABELS,
        device=device)

    return test_model(classifier, datasets['test'], data_loaders['test'], device)

In [9]:
if CCVAE not in result:
    result[CCVAE] = run_all_senario(main_gzsda, DOMAIN_SET)

## m0

In [10]:
def main_m0(args):
    set_seed(args)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    datasets, data_loaders = get_datesets_and_loaders(
        args=args,
        DOMAIN_SET=DOMAIN_SET,
        DATA_DIR=DATA_DIR,
        DATASET_DETAILS=DATASET_DETAILS)

    vae = get_trained_VAE(
        data_loaders=data_loaders,
        args=args,
        device=device)

    classifier = get_trained_classifier(
        data_loaders=data_loaders,
        vae=vae,
        NUM_LABELS=NUM_LABELS,
        device=device,
        change_policy_epoch=30)

    return test_model(classifier, datasets['test'], data_loaders['test'], device)

In [11]:
if our0 not in result:
    result[our0] = run_all_senario(main_m0, DOMAIN_SET)

## m1: seperate after encoder

In [12]:
def main_m1(args):
    set_seed(args)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    datasets, data_loaders = get_datesets_and_loaders(
        args=args,
        DOMAIN_SET=DOMAIN_SET,
        DATA_DIR=DATA_DIR,
        DATASET_DETAILS=DATASET_DETAILS)

    vae = get_trained_VAE_with_domain_classifier(
        data_loaders=data_loaders,
        args=args,
        device=device)
        
    classifier = get_trained_classifier(
        data_loaders=data_loaders,
        vae=vae,
        NUM_LABELS=NUM_LABELS,
        device=device,
        change_policy_epoch=30)

    return test_model(classifier, datasets['test'], data_loaders['test'], device)

In [13]:
if our_GRE not in result:
    result[our_GRE] = run_all_senario(main_m1, DOMAIN_SET)

## TUPL

In [14]:
def main_tupl(args):
    set_seed(args)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    acc_s, acc_u, h = run_tupl(
        data_root="./data/",
        dataset="xraybaggage20",
        source=args.sourceDomainIndex,
        target=args.targetDomainIndex,
        trial=args.trialIndex,
        seed=args.seed,
        device=device,
        quiet=True,
        return_model=False,
    )
    print('seen acc:{:2.4f}, unseen acc:{:2.4f}, H:{:2.4f}'.format(acc_s / 100, acc_u / 100, h / 100))
    return None, None, acc_s / 100.0, acc_u / 100.0

In [15]:
if tupl not in result:
    result[tupl] = run_all_senario(main_tupl, DOMAIN_SET)


## Merge results

In [16]:
with open(RESULT_OBJ_PATH, "w") as f:
    json.dump(result, f, indent=2)

In [17]:
# ignore our0
result.pop(our0, None)

{'regu -> xray': 'Seen:     74.19 ± 2.29\nUnseen:   33.84 ± 3.04\nH-mean:   46.23 ± 3.23',
 'xray -> regu': 'Seen:     87.35 ± 1.03\nUnseen:   54.64 ± 2.74\nH-mean:   67.02 ± 1.96'}

In [18]:
import pandas as pd
import re

rows = [(k, m, result[m][k]) for m in result for k in result[m]]
df = pd.DataFrame(rows, columns=['domain', 'method', 'values'])

def extract_metrics(text):
    matches = dict(re.findall(r'(\w+):\s+([\d.]+\s*±\s*[\d.]+)', text))
    return pd.Series(matches)

df[['seen', 'unseen', 'H-mean']] = df['values'].apply(extract_metrics)
df = df[['domain', 'method', 'seen', 'unseen', 'H-mean']]

df['method'] = pd.Categorical(df['method'], categories=[base, CCVAE, tupl, our0, our_GRE], ordered=True)
df = df.sort_values(['domain', 'method']).reset_index(drop=True)

df

,domain,method,seen,unseen,H-mean
0,regu -> xray,base,84.21 ± 2.22,2.56 ± 0.49,4.94 ± 0.92
1,regu -> xray,CCVAE,77.56 ± 2.11,29.50 ± 2.95,42.50 ± 3.44
2,regu -> xray,TUPL,81.00 ± 1.52,10.46 ± 1.36,18.37 ± 2.14
3,regu -> xray,our_GRE,72.31 ± 2.45,32.77 ± 1.47,45.05 ± 1.68
4,xray -> regu,base,94.66 ± 1.00,20.41 ± 4.23,32.73 ± 6.05
5,xray -> regu,CCVAE,90.67 ± 1.01,52.01 ± 2.86,65.88 ± 2.29
6,xray -> regu,TUPL,87.74 ± 1.67,30.38 ± 4.43,44.38 ± 4.84
7,xray -> regu,our_GRE,86.87 ± 0.75,55.23 ± 2.16,67.41 ± 1.54


In [19]:
df.to_csv(RESULT_CSV_PATH, index=False)